# MM-SafetyBench — Qwen2.5-VL-3B text-only baseline

This extension adds one text-only answer per question ID from an already completed run of the three image conditions. Complete the image run first. Put the authors' source ZIP next to the image run files in your private Drive folder. This notebook defaults to validating coverage without model inference.

For a generation run, use the archived T4 software versions, fill in your own approval record and GPU-minute limit, and enable `RUN_QWEN_TEXT`. The smoke output is separate from the full output. A full run needs `QWEN_TEXT_SMOKE = False`; preserve any partial run before retrying. See the README for scope and environment details.


In [ ]:
# *** -------
from collections import Counter
from hashlib import sha256
from pathlib import Path
from zipfile import ZipFile
import json
import os

from google.colab import drive
drive.mount('/content/drive')
PRIVATE = Path('/content/drive/MyDrive/MM-SafetyBench-Private')
RUNS = PRIVATE / 'Runs'
AUTHOR_ZIP = PRIVATE / 'MM-SafetyBench-main.zip'
QWEN_TEXT_PREFLIGHT = True
RUN_QWEN_TEXT = False
QWEN_TEXT_SNAPSHOT = None
QWEN_TEXT_SMOKE = True
OWNER_APPROVAL_RECORD = ''
GPU_MINUTES = None
AUTHOR_SHA = "801a891382a06b7c3b29083dd3844b6da6a3e0ef31d3dfdd9086e30b323f322b"
APPROVED_MANIFEST_SHA = "2ac5c293f5b50ac3c862e49f98091a37c020cbddeabee91d9be59e09e703b6d8"


In [ ]:
# *** -------
def read_saved_qwen(root):
    import hashlib

    manifest = json.loads((root / "manifest_PRIVATE.json").read_text())
    fingerprint = sha256(json.dumps(manifest, ensure_ascii=False, sort_keys=True).encode()).hexdigest()

    def identity(row):
        return tuple(row[name] for name in ("scenario", "id", "variant"))

    expected = {identity(row): row for row in manifest}
    aggregate = json.loads((root / "aggregate.json").read_text())
    if (len(manifest) != 5040 or len(expected) != 5040
            or fingerprint != (root / "manifest_sha256.txt").read_text().strip()
            or aggregate["manifest_sha256"] != fingerprint):
        raise ValueError("Saved manifest does not match its checksum and inventory")
    seen = set()
    counts = Counter()
    response_hash = hashlib.sha256()
    with (root / "responses_PRIVATE.jsonl").open("rb") as stream:
        for line in stream:
            response_hash.update(line)
            row = json.loads(line)
            k = identity(row)
            if (k not in expected or k in seen
                    or row["field"] != expected[k]["field"] or not row["response"].strip()):
                raise ValueError("Saved response failed its case check")
            seen.add(k)
            counts[row["variant"]] += 1
    if (seen != expected.keys() or aggregate["completed"] != len(seen)
            or aggregate["remaining"] != 0 or aggregate["status"] != "completed_pass"):
        raise ValueError("Saved response coverage disagrees with the aggregate")
    return expected, dict(completed=len(seen), by_carrier=dict(counts),
                          failure_attempts=aggregate["failure_attempts"],
                          manifest_sha256=fingerprint, response_sha256=response_hash.hexdigest())

def digest(path):
    with path.open("rb") as stream:
        return __import__("hashlib").file_digest(stream, "sha256").hexdigest()

def manifest_hash(rows):
    return sha256(json.dumps(rows, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def atomic_json(path, value):
    temp = path.with_suffix(".tmp")
    temp.write_text(json.dumps(value, indent=2))
    temp.replace(path)

def append_private(path, record):
    with path.open("a") as stream:
        stream.write(json.dumps(record, ensure_ascii=False) + "\n")
        stream.flush()
        os.fsync(stream.fileno())

In [ ]:
# *** -------
def qwen_text_cases(root):
    expected, _ = read_saved_qwen(root)
    cases = []
    with ZipFile(root / AUTHOR_ZIP.name) as author:
        for name in sorted(author.namelist()):
            path = Path(name)
            if path.parent.name != "processed_questions" or path.suffix != ".json":
                continue
            for item, fields in json.loads(author.read(name)).items():
                question = fields["Question"]
                if not isinstance(question, str) or not question.strip():
                    raise ValueError("Invalid original question")
                cases.append(dict(scenario=path.stem, id=item, variant="TEXT",
                                  field="Question", question=question))
    keys = {(row["scenario"], row["id"], carrier) for row in cases
            for carrier in ("SD", "TYPO", "SD_TYPO")}
    if len(cases) != 1680 or len({row["scenario"] for row in cases}) != 13 or keys != expected.keys():
        raise ValueError("Text baseline does not match saved image case IDs")
    return cases


def run_qwen_text(rows):
    import time

    from huggingface_hub import snapshot_download, try_to_load_from_cache

    model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
    revision = "66285546d2b821cf421d4f5eb2576359d3770cd3"
    manifest_sha = manifest_hash(rows)
    if not (OWNER_APPROVAL_RECORD and GPU_MINUTES and GPU_MINUTES > 0
            and APPROVED_MANIFEST_SHA == manifest_sha):
        raise RuntimeError("Approval, GPU minutes, and the frozen manifest hash are required")

    import torch
    import transformers
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

    previous = json.loads((PRIVATE / "aggregate.json").read_text())
    if (torch.__version__ != previous["torch"]
            or transformers.__version__ != previous["transformers"]
            or not torch.cuda.is_available() or "T4" not in torch.cuda.get_device_name(0)):
        raise RuntimeError("The archived T4 software configuration is required")
    deadline = time.monotonic() + GPU_MINUTES * 60
    mode = "smoke" if QWEN_TEXT_SMOKE else "full"
    if QWEN_TEXT_SMOKE:
        rows = [dict(scenario="benign_smoke", id="1", variant="TEXT", field="Question",
                     question="Name two colors found in a rainbow.")]
    suffix = "_Smoke" if QWEN_TEXT_SMOKE else ""
    out = RUNS / f"MM_SafetyBench_Qwen25_Text{suffix}_Private"
    out.mkdir(parents=True, exist_ok=False)
    status = dict(model=model_id, revision=revision, torch=previous["torch"],
                  transformers=previous["transformers"], precision="float16",
                  max_new_tokens=256, do_sample=False, planned=len(rows),
                  completed=0, failures=0, status="running", approval=OWNER_APPROVAL_RECORD,
                  manifest_sha256=manifest_sha, mode=mode)
    atomic_json(out / "aggregate.json", status)
    try:
        print(f"Qwen {mode}: preparing the pinned checkpoint.", flush=True)
        if QWEN_TEXT_SNAPSHOT:
            snapshot = Path(QWEN_TEXT_SNAPSHOT).resolve()
            cached = try_to_load_from_cache(model_id, "config.json", revision=revision)
            if not isinstance(cached, str) or Path(cached).parent.resolve() != snapshot:
                raise RuntimeError("Snapshot does not match the archived Qwen revision")
        else:
            snapshot = Path(snapshot_download(model_id, revision=revision))
        if time.monotonic() >= deadline:
            status["status"] = "budget_stop"
            return
        print("Loading Qwen in FP16.", flush=True)
        processor = AutoProcessor.from_pretrained(str(snapshot), local_files_only=True)
        model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            str(snapshot), torch_dtype=torch.float16, device_map="auto", local_files_only=True).eval()
        for row in rows:
            if time.monotonic() >= deadline:
                status["status"] = "budget_stop"
                break
            identity = {name: row[name] for name in ("scenario", "id", "variant", "field")}
            try:
                messages = [{"role": "user", "content": [{"type": "text", "text": row["question"]}]}]
                prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                inputs = processor(text=[prompt], return_tensors="pt").to(model.device)
                with torch.inference_mode():
                    tokens = model.generate(**inputs, do_sample=False, max_new_tokens=256)
                answer = processor.batch_decode(tokens[:, inputs.input_ids.shape[1]:],
                                                skip_special_tokens=True)[0]
                if not answer.strip():
                    raise ValueError("Empty completion")
                append_private(out / "responses_PRIVATE.jsonl", dict(identity, response=answer))
                status["completed"] += 1
                if QWEN_TEXT_SMOKE or status["completed"] % 50 == 0:
                    atomic_json(out / "aggregate.json", status)
                    print(f"Saved {status['completed']:,}/{len(rows):,} answers.", flush=True)
            except Exception as exc:
                append_private(out / "failures_PRIVATE.jsonl", dict(identity, error_type=type(exc).__name__))
                status.update(failures=1, status="case_failure")
                break
        else:
            status["status"] = "completed_unscored"
    finally:
        if status["status"] == "running":
            status["status"] = "interrupted_or_setup_failure"
        status["remaining"] = status["planned"] - status["completed"]
        atomic_json(out / "aggregate.json", status)
        print({name: status[name] for name in ("status", "planned", "completed", "failures", "remaining")})


if QWEN_TEXT_PREFLIGHT or RUN_QWEN_TEXT:
    print("Matching text-only questions to the archived Qwen image run.", flush=True)
    if not PRIVATE.is_dir():
        raise FileNotFoundError(f"Mount college Drive and verify {PRIVATE}")
    if digest(AUTHOR_ZIP) != AUTHOR_SHA:
        raise ValueError("Unexpected author ZIP")
    text_cases = qwen_text_cases(PRIVATE)
    print(dict(text_cases=len(text_cases), manifest_sha256=manifest_hash(text_cases)))
    if RUN_QWEN_TEXT:
        run_qwen_text(text_cases)
    else:
        print("Input checks complete. This preflight did not generate answers.")
